# Mesh Tutorial 9: Mesh Generation

PhysicsNeMo-Mesh *generates* simulation-ready volume meshes two ways, plus a surface extractor:

| | `tessellation.fill_interior` | `generate.mesh_implicit_domain` | `generate.marching_cubes` |
|---|---|---|---|
| input | closed boundary `Mesh` | implicit function $\varphi(x)$ | 3D scalar grid |
| output | volume `Mesh` | volume `Mesh`, **any dimension** | *surface* `Mesh` |
| boundary | **exact** (bit-identical vertices) | $O(h^2)$ approximation | isosurface |
| quality | **guaranteed** min angle (2D) | optimized distributions | n/a |
| special powers | provenance data, bitwise determinism | GPU scale, robustness-by-construction, differentiable | speed |

## What You'll Learn

1. Filling a multiply-connected boundary `Mesh` exactly, and using provenance for per-boundary BCs
2. Verifying the minimum-angle *guarantee* across resolutions
3. Meshing implicit domains: CSG, raw level sets, and what "cannot fail" means
4. Pinning sharp corners with `feature_points` (and what happens if you don't)
5. Tetrahedralizing 3D implicit domains
6. Differentiable meshing: gradients of mesh quantities w.r.t. shape parameters

## Section 1: Exact-boundary filling with `fill_interior`

Give it a boundary edge `Mesh` — loops in any order and orientation; holes, multiple components, and islands-inside-holes are resolved automatically. A gear with a star hole and an island:

In [ ]:
import math

import matplotlib.pyplot as plt
import torch

from physicsnemo.mesh import Mesh
from physicsnemo.mesh.tessellation import fill_interior


def loop(f, n, offset):
    t = torch.arange(n, dtype=torch.float64) / n * 2 * math.pi
    i = torch.arange(n)
    return f(t), torch.stack([i, (i + 1) % n], dim=1) + offset


def gear(t, r0=1.0, teeth=9, depth=0.13):
    r = r0 + depth * torch.tanh(4 * torch.sin(teeth * t))
    return torch.stack([r * torch.cos(t), r * torch.sin(t)], dim=1)


def star(t, r0=0.42, k=5, a=0.3):
    r = r0 * (1 + a * torch.cos(k * t))
    return torch.stack([r * torch.cos(t), r * torch.sin(t)], dim=1)


p1, e1 = loop(gear, 240, 0)
p2, e2 = loop(star, 72, 240)
p3, e3 = loop(
    lambda t: torch.stack([0.1 * torch.cos(t), 0.1 * torch.sin(t)], 1), 16, 312
)
boundary = Mesh(points=torch.cat([p1, p2, p3]), cells=torch.cat([e1, e2, e3]))

h = 0.045
filled = fill_interior(
    boundary,
    max_cell_size=math.sqrt(3) / 4 * h * h,
    smooth_iterations=3,
    provenance=True,  # opt-in: attaches boundary_marker / source_point
)

pts = filled.points.numpy()
fig, ax = plt.subplots(figsize=(7.5, 7))
ax.triplot(pts[:, 0], pts[:, 1], filled.cells.numpy(), lw=0.3, color="tab:blue")
bp = boundary.points.numpy()
ax.plot(bp[:, 0], bp[:, 1], ".", ms=2.5, color="tab:red")
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("fill_interior: gear + star hole + island, one edge Mesh in")
plt.show()

print(
    "input vertices preserved bit-identically:",
    bool(torch.equal(filled.points[: boundary.n_points], boundary.points)),
)

### Provenance: per-boundary data without bookkeeping

`point_data["source_point"]` maps every inherited vertex back to its input index, so per-boundary conditions survive meshing. Tag each input loop, then recover the tags on the output:

In [ ]:
# Which input loop did each input vertex belong to?
loop_id = torch.cat([torch.full((n,), k) for k, n in enumerate([240, 72, 16])])

src = filled.point_data["source_point"]
out_loop = torch.full((filled.n_points,), -1)
out_loop[src >= 0] = loop_id[src[src >= 0]]

fig, ax = plt.subplots(figsize=(7.5, 7))
ax.triplot(pts[:, 0], pts[:, 1], filled.cells.numpy(), lw=0.25, color="0.8")
for k, (name, c) in enumerate(
    [("outer gear", "tab:blue"), ("star hole", "tab:red"), ("island", "tab:green")]
):
    sel = (out_loop == k).numpy()
    ax.plot(pts[sel, 0], pts[sel, 1], ".", ms=4, color=c, label=name)
ax.legend(loc="upper right")
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("provenance: output vertices classified by source loop")
plt.show()

### The guarantee, verified across resolutions

Ruppert refinement *proves* the angle bound — it is not a tendency. Histograms of the per-triangle minimum angle at three resolutions; nothing ever crosses the 30° line:

In [ ]:
def min_angles(m):
    p = m.points[m.cells]
    out = None
    for i in range(3):
        u = p[:, (i + 1) % 3] - p[:, i]
        v = p[:, (i + 2) % 3] - p[:, i]
        cos = (u * v).sum(-1) / (u.norm(dim=-1) * v.norm(dim=-1))
        a = torch.rad2deg(torch.arccos(cos.clamp(-1, 1)))
        out = a if out is None else torch.minimum(out, a)
    return out


import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, hh in zip(axes, [0.12, 0.06, 0.03]):
    m = fill_interior(boundary, max_cell_size=math.sqrt(3) / 4 * hh * hh)
    ang = min_angles(m).numpy()
    ax.hist(ang, bins=np.linspace(20, 60, 41), color="tab:blue")
    ax.axvline(30, color="tab:red", ls="--", lw=1)
    ax.set_title(f"h={hh}: {m.n_cells} tris, min {ang.min():.2f}°")
    ax.set_xlabel("min angle [deg]")
axes[0].set_ylabel("count")
fig.suptitle("guaranteed ≥ 30° at every resolution", y=1.05)
plt.show()

## Section 2: Implicit domains with `mesh_implicit_domain`

No boundary discretization at all — just a function that is negative inside. It **cannot fail to return a valid mesh**: every optimization step is validity-gated, and a coverage guard raises if your geometry has features finer than `h` (instead of silently dropping them). A gallery:

In [ ]:
from physicsnemo.mesh.generate import (
    mesh_implicit_domain,
    sdf_box,
    sdf_difference,
    sdf_sphere,
    sdf_union,
)

holes = [
    sdf_sphere([0.55 * i - 0.55, 0.55 * j - 0.55], 0.16)
    for i in range(3)
    for j in range(3)
]
plate = sdf_difference(sdf_box([-1, -1], [1, 1]), sdf_union(*holes))


def blob(x):  # a raw level set -- NOT a distance function; still fine
    r = x.norm(dim=-1)
    th = torch.atan2(x[..., 1], x[..., 0])
    return (r - (0.55 + 0.16 * torch.cos(6 * th))) * 2.7


dumbbell = sdf_union(
    sdf_sphere([-0.45, 0.0], 0.38),
    sdf_sphere([0.45, 0.0], 0.38),
    sdf_box([-0.45, -0.1], [0.45, 0.1]),
)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (phi, box, hh, name) in zip(
    axes,
    [
        (plate, ([-1, -1], [1, 1]), 0.05, "CSG plate, 9 holes"),
        (blob, ([-0.85, -0.85], [0.85, 0.85]), 0.05, "raw level set"),
        (dumbbell, ([-1, -0.6], [1, 0.6]), 0.05, "near-tangent union"),
    ],
):
    m = mesh_implicit_domain(phi, box, hh)
    p = m.points.numpy()
    ax.triplot(p[:, 0], p[:, 1], m.cells.numpy(), lw=0.3, color="tab:blue")
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(name)
plt.show()

### Sharp corners: rounded by default, exact with `feature_points`

Implicit meshing has nothing pulling a vertex *into* a corner tip, so corners round off at the $h$ scale. Pin them and they are interpolated exactly (a vertex is topologically inserted at each):

In [ ]:
bitten = sdf_difference(sdf_sphere([0.0, 0.0], 0.7), sdf_box([0.2, -0.4], [1.0, 0.4]))
xt = math.sqrt(0.7**2 - 0.4**2)
corners = torch.tensor(
    [[0.2, -0.4], [0.2, 0.4], [xt, 0.4], [xt, -0.4]], dtype=torch.float64
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharex=True, sharey=True)
for ax, fp, title in [
    (axes[0], None, "default: corners rounded"),
    (axes[1], corners, "feature_points: corners exact"),
]:
    m = mesh_implicit_domain(
        bitten, ([-1, -1], [1, 1]), 0.06, feature_points=fp, max_coverage_gap_h=None
    )
    p = m.points.numpy()
    ax.triplot(p[:, 0], p[:, 1], m.cells.numpy(), lw=0.3, color="tab:blue")
    ax.plot(corners[:, 0], corners[:, 1], "*", ms=12, color="tab:red")
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlim(0.0, 0.85)
    ax.set_ylim(0.15, 0.6)
plt.show()

### 3D (and beyond): the same code path

`mesh_implicit_domain` is dimension-generic — the dimension is just `len(bounds[0])`. A tet-meshed spherical shell, cut away; on CUDA this scales to ~10⁵–10⁶ cells in seconds (`device="cuda"`):

In [ ]:
shell = sdf_difference(sdf_sphere([0.0] * 3, 0.8), sdf_sphere([0.0] * 3, 0.4))
m3, diag = mesh_implicit_domain(shell, ([-1] * 3, [1] * 3), 0.09, full_output=True)
print(
    f"{diag['n_cells']} tets | watertight: {diag['boundary_closed_manifold']} "
    f"| sliver fraction: {100 * diag['sliver_fraction']:.1f}% "
    f"| coverage gap: {diag['coverage_gap_h']:.2f} h"
)

cen = m3.points[m3.cells].mean(dim=1)
keep = m3.cells[cen[:, 0] < 0]
faces = torch.cat(
    [keep[:, [0, 1, 2]], keep[:, [0, 1, 3]], keep[:, [0, 2, 3]], keep[:, [1, 2, 3]]]
)
p3 = m3.points.numpy()
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_trisurf(
    p3[:, 0],
    p3[:, 1],
    faces.numpy(),
    p3[:, 2],
    color="tab:blue",
    edgecolor="white",
    lw=0.05,
    alpha=0.95,
)
ax.set_box_aspect((1, 1, 1))
ax.set_title("3D shell, cutaway at x = 0")
plt.show()

## Section 3: Differentiable meshing

`refit_mesh_to_implicit` re-projects a mesh's boundary onto $\varphi = 0$ with graph-preserving Newton steps at fixed topology — so gradients flow from any mesh-derived quantity back to shape parameters. Here: mesh a disk once, then sweep its radius differentiably and compare $dV/dr$ from autograd with the analytic perimeter $2\pi r$:

In [ ]:
from physicsnemo.mesh.generate import refit_mesh_to_implicit


def cell_volumes(m):
    p = m.points[m.cells]
    e1, e2 = p[:, 1] - p[:, 0], p[:, 2] - p[:, 0]
    return 0.5 * (e1[:, 0] * e2[:, 1] - e1[:, 1] * e2[:, 0])


base = mesh_implicit_domain(sdf_sphere([0.0, 0.0], 0.7), ([-1, -1], [1, 1]), 0.06)

radii = torch.linspace(0.6, 0.8, 9, dtype=torch.float64)
dV = []
for r0 in radii:
    r = r0.clone().requires_grad_(True)
    refit = refit_mesh_to_implicit(base, lambda x: x.norm(dim=-1) - r)
    (g,) = torch.autograd.grad(cell_volumes(refit).sum(), r)
    dV.append(float(g))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(
    radii,
    [2 * math.pi * float(r) for r in radii],
    "-",
    color="0.6",
    label="analytic  $2\pi r$",
)
ax.plot(radii, dV, "o", color="tab:blue", label="autograd through the mesh")
ax.set_xlabel("radius r")
ax.set_ylabel("dV/dr")
ax.legend()
ax.set_title("meshing as a differentiable layer")
plt.show()

## Recap

* **`fill_interior`** when the boundary discretization is the contract: exact vertices, guaranteed angles, provenance `point_data`, bitwise determinism. 2D today; 3D raises `NotImplementedError` pending exact boundary recovery.
* **`mesh_implicit_domain`** when geometry is implicit or 3D+/GPU-scale: robust by construction, coverage-guarded, `feature_points` for corners, `refit_mesh_to_implicit` for shape gradients.
* **`marching_cubes`** when you want the isosurface itself, not a volume.

### Where to next

* **Tutorial 5** for the quality metrics used throughout; **Tutorial 8** for I/O and interop.